🧠 Purpose:
This script converts your questions_and_answers.json (in OpenAI Chat format) into the Alpaca-style .jsonl format required for instruction-tuned fine-tuning (QLoRA).

It is a required preprocessing step for:

qlora_finetune_sft.ipynb (Pipeline P1)

Step 1: Import Libraries
Loads Python libraries for:

File handling (os, Path)

JSON parsing

Progress visualization (tqdm)



In [1]:
# 📌 Step 1: Import Required Libraries
import json
import os
from pathlib import Path
from tqdm import tqdm

Step 2: Define Paths
input_file: your questions_and_answers.json in OpenAI format ({"messages": [...]})

output_file: target Alpaca-format .jsonl for use in QLoRA training

In [2]:
# 📌 Step 2: Define Paths
input_file = Path("../data/questions_and_answers.json")
output_file = Path("../data/sft_alpaca.jsonl")

Step 3: Load Dataset
Opens and reads the OpenAI-format chat data

Assumes each item has messages with "role": "user" and "role": "assistant"

In [3]:
# 📌 Step 3: Load OpenAI Chat-Style Dataset
with open(input_file, 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

Step 4: Convert to Alpaca Format
Iterates through each example

Extracts:

User question → instruction

Assistant reply → output

Leaves input field blank (since all your Q&A pairs are single-turn)

✔ Adds each to a list as a dict

In [4]:
# 📌 Step 4: Convert to Alpaca Format
converted_data = []

for sample in tqdm(qa_data):
    messages = sample.get("messages", [])
    
    user_msg = next((m["content"] for m in messages if m["role"] == "user"), "").strip()
    assistant_msg = next((m["content"] for m in messages if m["role"] == "assistant"), "").strip()

    if user_msg and assistant_msg:
        converted_data.append({
            "instruction": user_msg,
            "input": "",
            "output": assistant_msg
        })


100%|██████████| 635/635 [00:00<00:00, 632482.32it/s]


Step 5: Save as JSONL
Writes out each converted entry to sft_alpaca.jsonl, one JSON object per line

Prints success message with the total count

In [5]:
# 📌 Step 5: Save as JSONL
with open(output_file, 'w', encoding='utf-8') as f:
    for item in converted_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"✅ Converted {len(converted_data)} entries to: {output_file}")

✅ Converted 635 entries to: ..\data\sft_alpaca.jsonl


✅ Suggestions

Area	Suggestion
File name	✅ prepare_supervised_dataset.ipynb is perfect
Validation	Add a check to ensure both user and assistant exist in each message
Output preview	Print a few sample converted lines for visual sanity check
Logging	Save log.txt for number of skipped/invalid entries (optional)
✅ Pipeline Role
This script feeds into:


Component	Used by
sft_alpaca.jsonl	✅ qlora_finetune_sft.ipynb
Pipeline	✅ P1 (SFT), P3 (DAPT+SFT), P4 (Full)